In [1]:
# Imports
import sys
sys.path.append("Program")

from backtest import *
import concurrent.futures
import datetime as dt
from functools import partial
from fundamentals import *
from helper_functions import modify_current_date, get_df, get_infix
import matplotlib.pyplot as plt
import multiprocessing
import numpy as np
import pandas as pd
from plot import *
from technicals import *
from tqdm import tqdm

# Start of the program
start = dt.datetime.now()

# Variables
all_stocks = False
period_short = 60
period_long = 252
RS = 90
factors = [0.1, 0.55, 0.35]
backtest = False

# Index
index_name = "^GSPC"
index_dict = {"^HSI": "Hang Seng Index", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite"}

# Get the infix
infix = get_infix("^GSPC", index_dict, all_stocks) 

# Modify the current date
current_date = modify_current_date(start, index_name)

In [2]:
# Index
indices = ["^HSI", "^GSPC", "^IXIC", "QLD", "TQQQ"]
index_dict = {"^HSI": "Hang Seng Index", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite", "QLD": "ProShares Ultra QQQ", "TQQQ": "ProShares UltraPro QQQ"}

# Momentum ETFs
etfs_dict = {
    "MTUM": "iShares MSCI USA Momentum Factor ETF",
    "IMTM": "iShares MSCI Intl Momentum Factor ETF",
    "SMLF": "iShares U.S. Small‑Cap Equity Factor ETF",
    "IUSV": "iShares S&P U.S. Value ETF",
    "IJJ": "iShares S&P Mid-Cap 400 Value ETF",
    "IJS": "iShares S&P Small-Cap 600 Value ETF",
    "EFV": "iShares MSCI EAFE Value ETF",
    "SPMO": "Invesco S&P 500 Momentum ETF",
    "XMMO": "Invesco S&P MidCap Momentum ETF",
    "XSMO": "Invesco S&P SmallCap Momentum ETF",
    "PDP": "Invesco Dorsey Wright Momentum ETF",
    "IDMO": "Invesco S&P Intl Developed Momentum ETF",
    "RPV": "Invesco S&P 500 Pure Value ETF",
    "SPHD": "Invesco S&P 500 High Dividend Low Volatility ETF",
    "VFMO": "Vanguard U.S. Momentum Factor ETF",
    "VYM": "Vanguard High Dividend Yield ETF",
    "VOE": "Vanguard S&P Mid-Cap Value ETF",
    "VONV": "Vanguard Russell 1000 Value ETF",
    "VOOV": "Vanguard S&P 500 Value ETF",
    "MGV": "Vanguard Mega Cap Value ETF",
    "VBR": "Vanguard Small-Cap Value ETF",
    "VTV": "Vanguard Value ETF",
    "AVLV": "Avantis U.S. Large Cap Value ETF",
    "AVUV": "Avantis U.S. Small Cap Value ETF",
    "JMOM": "JPMorgan U.S. Momentum Factor ETF",
    "GLOV": "Activebeta World Low Vol Plus Equity ETF",
    "DWMF": "WisdomTree International Multifactor ETF",
    "AFLG": "First Trust Active Factor Large Cap Intl ETF",
    "CGDV": "Capital Group Dividend Value ETF",
    "SCHD": "Schwab U.S. Dividend Equity ETF",
    "3070.HK": "Ping An of China CSI HK Dividend ETF"
}
etfs = list(etfs_dict.keys())

In [3]:
# Create an empty DataFrame to store the data of the ETFs
etfs_data = pd.DataFrame(columns=[
    "Symbol", "Establishment Date", "1 Year CAGR (%)", "3 Year CAGR (%)", 
    "5 Year CAGR (%)", "10 Year CAGR (%)", "20 Year CAGR (%)", "Max Period CAGR (%)", 
    "Max Period Annual Volatility (%)", "Max Period Sharpe", 
    "Max Period Sortino", "Max Period MDD (%)", "Max Period Calmar", 
    "5 Year Annual Volatility (%)", "5 Year Sharpe", "5 Year Sortino", 
    "5 Year MDD (%)", "5 Year Calmar", "5 Year Peak Date", 
    "5 Year Recovery Date", "5 Year Recovery Period (days)"
])

# Loop through each ETF in the indices and momentum ETFs
for etf in tqdm(indices + etfs):
    # Get the name of the ETF
    etf_name = {**index_dict, **etfs_dict}.get(etf, etf)
    etfs_data.loc[etf_name, "Symbol"] = etf

    # Get the dataframe for the ETF
    df = get_df(etf, current_date, max_period=True, adj=True)
    
    # Get the establishment date of the ETF
    establishment_date = df.index[0].date()
    etfs_data.loc[etf_name, "Establishment Date"] = establishment_date
    
    # Calculate CAGR values for different periods
    for period, label in zip([252, 252 * 3, 252 * 5, 252 * 10, 252 * 20], 
                             ["1 Year CAGR (%)", "3 Year CAGR (%)", "5 Year CAGR (%)", "10 Year CAGR (%)", "20 Year CAGR (%)"]):
        if len(df) >= period:
            start_price = df["Close"].iloc[- period]
            end_price = df["Close"].iloc[- 1]
            cagr = ((end_price / start_price) ** (1 / (period / 252)) - 1) * 100
            etfs_data.loc[etf_name, label] = cagr

    # Calculate CAGR of maximum period
    start_price = df["Close"].iloc[0]
    end_price = df["Close"].iloc[- 1]
    max_period_cagr = ((end_price / start_price) ** (1 / (len(df) / 252)) - 1) * 100
    etfs_data.loc[etf_name, "Max Period CAGR (%)"] = max_period_cagr

    # Calculate percent change and cumulative return
    df["Percent Change"] = df["Close"].pct_change().dropna()
    df["Cumulative Return"] = (1 + df["Percent Change"]).cumprod()
    daily_returns = df["Percent Change"]
    cumulative_returns = df["Cumulative Return"]
        
    # Calculate annual volatility
    annual_volatility = daily_returns.std() * np.sqrt(252) * 100
    etfs_data.loc[etf_name, "Max Period Annual Volatility (%)"] = annual_volatility

    # Calculate 5 year annual volatility
    if len(df) >= 252 * 5:
        daily_returns_5y = daily_returns.iloc[- 252 * 5:]
        annual_volatility_5y = daily_returns_5y.std() * np.sqrt(252) * 100
        etfs_data.loc[etf_name, "5 Year Annual Volatility (%)"] = annual_volatility_5y

    # Calculate Sharpe ratio
    risk_free_rate = 0.03  # Assuming a risk-free rate of 3%
    sharpe_ratio = (daily_returns.mean() * 252 - risk_free_rate) / (daily_returns.std() * np.sqrt(252))
    etfs_data.loc[etf_name, "Max Period Sharpe"] = sharpe_ratio

    # Calculate 5 year Sharpe ratio
    if len(df) >= 252 * 5:
        sharpe_ratio_5y = (daily_returns_5y.mean() * 252 - risk_free_rate) / (daily_returns_5y.std() * np.sqrt(252))
        etfs_data.loc[etf_name, "5 Year Sharpe"] = sharpe_ratio_5y

    # Calculate Sortino ratio
    negative_returns = daily_returns[daily_returns < 0]
    sortino_ratio = (daily_returns.mean() * 252 - risk_free_rate) / (negative_returns.std() * np.sqrt(252))
    etfs_data.loc[etf_name, "Max Period Sortino"] = sortino_ratio

    # Calculate 5 year Sortino ratio
    if len(df) >= 252 * 5:
        negative_returns_5y = daily_returns_5y[daily_returns_5y < 0]
        sortino_ratio_5y = (daily_returns_5y.mean() * 252 - risk_free_rate) / (negative_returns_5y.std() * np.sqrt(252))
        etfs_data.loc[etf_name, "5 Year Sortino"] = sortino_ratio_5y

    # Calculate maximum drawdown
    df["Peak"] = df["Cumulative Return"].cummax()
    df["Drawdown"] = (df["Cumulative Return"] - df["Peak"]) / df["Peak"]
    drawdowns = df["Drawdown"]
    mdd = drawdowns.min() * 100
    mdd_date = drawdowns.idxmin()
    etfs_data.loc[etf_name, "Max Period MDD (%)"] = mdd

    # Calculate 5 year maximum drawdown
    if len(df) >= 252 * 5:
        drawdowns_5y = drawdowns[- 252 * 5:]
        mdd_5y = drawdowns_5y.min() * 100
        etfs_data.loc[etf_name, "5 Year MDD (%)"] = mdd_5y
        mdd_date_5y = drawdowns_5y.idxmin()

        # Find the peak date before the maximum drawdown
        peak_mask_5y = (df.index <= mdd_date_5y) & (df["Cumulative Return"] == df["Peak"])
        peak_date_5y = df.index[peak_mask_5y][-1]
        etfs_data.loc[etf_name, "5 Year Peak Date"] = peak_date_5y.date()

        # Find the recovery date after the maximum drawdown
        rec_mask_5y = (df.index > mdd_date_5y) & (df["Cumulative Return"] >= df.loc[peak_date_5y, "Cumulative Return"])
        rec_date_5y = df.index[rec_mask_5y][0] if len(df.index[rec_mask_5y]) > 0 else None
        if rec_date_5y is not None:
            etfs_data.loc[etf_name, "5 Year Recovery Date"] = rec_date_5y.date()
            rec_period_5y = (rec_date_5y - peak_date_5y).days
            etfs_data.loc[etf_name, "5 Year Recovery Period (days)"] = rec_period_5y

        # Calculate the two most significant maximum drawdown in the 5 year period
        drawdowns_5y = drawdowns_5y.sort_values()
        mdd_5y = drawdowns_5y.iloc[0] * 100

    # Calculate Calmar ratio
    calmar_ratio = (daily_returns.mean() * 252 - risk_free_rate) / abs(mdd / 100)
    etfs_data.loc[etf_name, "Max Period Calmar"] = calmar_ratio

    # Calculate 5 year Calmar ratio
    if len(df) >= 252 * 5:
        calmar_ratio_5y = (daily_returns_5y.mean() * 252 - risk_free_rate) / abs(mdd_5y / 100)
        etfs_data.loc[etf_name, "5 Year Calmar"] = calmar_ratio_5y

100%|██████████| 36/36 [00:22<00:00,  1.62it/s]


In [4]:
etfs_data

,Symbol,Establishment Date,1 Year CAGR (%),3 Year CAGR (%),5 Year CAGR (%),10 Year CAGR (%),20 Year CAGR (%),Max Period CAGR (%),Max Period Annual Volatility (%),Max Period Sharpe,...,Max Period MDD (%),Max Period Calmar,5 Year Annual Volatility (%),5 Year Sharpe,5 Year Sortino,5 Year MDD (%),5 Year Calmar,5 Year Peak Date,5 Year Recovery Date,5 Year Recovery Period (days)
Hang Seng Index,^HSI,1986-12-31,43.118342,4.233247,0.268907,-1.114846,3.175948,6.253555,25.687672,0.249946,...,-65.18186,0.098502,25.140103,0.017034,0.025372,-55.700772,0.007688,2018-01-26,NaN,NaN
S&P 500,^GSPC,1927-12-30,17.716736,16.901698,14.422157,11.618958,8.601805,6.244789,18.96701,0.256277,...,-86.189579,0.056397,17.358492,0.692116,0.949655,-25.425097,0.472529,2022-01-03,2024-01-19,746
NASDAQ Composite,^IXIC,1971-02-05,21.715033,20.51473,14.597081,14.997342,12.091401,10.319842,20.171788,0.439289,...,-77.932386,0.113704,23.148009,0.56769,0.797844,-36.39528,0.36106,2021-11-19,2024-02-29,832
ProShares Ultra QQQ,QLD,2006-06-21,33.580794,36.057122,24.371531,28.346942,NaN,24.246458,44.258869,0.645296,...,-83.128873,0.343564,46.6908,0.62714,0.88379,-63.684488,0.459793,2021-11-19,2024-05-24,917
ProShares UltraPro QQQ,TQQQ,2010-02-11,37.537237,43.827746,25.847419,33.139363,NaN,41.644037,61.464763,0.828741,...,-81.659847,0.623787,69.422285,0.626079,0.880902,-81.659847,0.532255,2021-11-19,2024-12-04,1111
iShares MSCI USA Momentum Factor ETF,MTUM,2013-04-18,29.912203,22.19971,12.603957,13.860455,NaN,14.924148,19.649905,0.654157,...,-34.082294,0.377149,21.276389,0.513785,0.710335,-32.284548,0.338598,2021-11-03,2024-03-04,852
iShares MSCI Intl Momentum Factor ETF,IMTM,2015-01-27,21.436892,18.085557,9.588287,8.237926,NaN,8.04296,20.249646,0.334244,...,-30.68315,0.220587,17.354452,0.443279,0.658076,-30.68315,0.250719,2021-11-08,2024-02-22,836
iShares U.S. Small‑Cap Equity Factor ETF,SMLF,2015-04-30,12.88226,13.717246,15.390233,9.921422,NaN,10.348345,21.870191,0.42331,...,-41.892201,0.220992,21.533849,0.644185,1.004907,-26.277099,0.527904,2024-11-25,NaN,NaN
iShares S&P U.S. Value ETF,IUSV,2000-08-04,9.809196,14.56995,14.706888,10.43442,8.540572,8.175623,19.23624,0.349112,...,-60.183502,0.111585,15.340498,0.791138,1.160874,-17.951551,0.676067,2022-04-20,2023-02-01,287
iShares S&P Mid-Cap 400 Value ETF,IJJ,2000-07-28,10.572099,10.939499,15.371651,9.220823,8.81466,10.127905,21.971386,0.412956,...,-58.003723,0.156425,20.609039,0.671723,1.057174,-23.42545,0.590962,2020-01-16,2020-12-04,323


In [5]:
# Save the data as a CSV file
result_folder = "Result"
filename = os.path.join(result_folder, "etfs_comparison.csv")
etfs_data.to_csv(filename)